# 03. Export features and report usable samples

Compute the existing feature recipes, retain the per-parcel cap and locked partitions, and save separate development and CropHarvest external-test packages. The resulting tables show how many fields and pixels passed, separately by crop and partition. No model fitting or test scoring occurs.


**Input:** Verified windows.json from Preprocessing 02. **Output:** `features.json plus separate development/test features`. **Next:** Training 01.

[Run order and reruns](../../docs/workflow.md).


In [1]:
from pathlib import Path
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import preprocessing_workflow as workflow
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows import seasons

window_input = ""  # Optional exact windows.json path from new notebook 02; blank uses its verified pointer.


## Export model inputs

Matched fixed/aligned cohorts retain identical locations. Full cohorts report their own coverage. Budgets are collection targets, not accuracy criteria; unmet budgets and absent crops remain visible. Do not add counts across cohorts.


In [2]:
feature_handoff = workflow.export_features(root, window_input or None)
usable_samples = seasons.read_table(feature_handoff.parent / "usable_samples.csv")
for cohort in ("fixed_full", "aligned_full", "fixed_matched", "aligned_matched"):
    show_table([r for r in usable_samples if r["cohort"] == cohort], title=f"Usable samples: {cohort}")


Cohort,Partition,Crop,Usable parcel groups,Usable pixels,Target parcels,Target pixels,Parcel shortfall,Pixel shortfall,Budget met
fixed_full,Training,Carrot,269,4851,60,1200,0,0,True
fixed_full,Training,Cabbage,422,8328,60,1200,0,0,True
fixed_full,Training,Potato,358,5120,60,1200,0,0,True
fixed_full,Validation,Carrot,94,1764,20,400,0,0,True
fixed_full,Validation,Cabbage,124,2462,20,400,0,0,True
fixed_full,Validation,Potato,147,2064,20,400,0,0,True
fixed_full,Reserved test,Carrot,21,65,20,400,0,335,False
fixed_full,Reserved test,Cabbage,38,351,20,400,0,49,False
fixed_full,Reserved test,Potato,32,134,20,400,0,266,False


Cohort,Partition,Crop,Usable parcel groups,Usable pixels,Target parcels,Target pixels,Parcel shortfall,Pixel shortfall,Budget met
aligned_full,Training,Carrot,107,1869,60,1200,0,0,True
aligned_full,Training,Cabbage,63,1190,60,1200,0,10,False
aligned_full,Training,Potato,87,1277,60,1200,0,0,True
aligned_full,Validation,Carrot,31,580,20,400,0,0,True
aligned_full,Validation,Cabbage,20,383,20,400,0,17,False
aligned_full,Validation,Potato,43,614,20,400,0,0,True
aligned_full,Reserved test,Carrot,20,62,20,400,0,338,False
aligned_full,Reserved test,Cabbage,12,95,20,400,8,305,False
aligned_full,Reserved test,Potato,22,87,20,400,0,313,False


Cohort,Partition,Crop,Usable parcel groups,Usable pixels,Target parcels,Target pixels,Parcel shortfall,Pixel shortfall,Budget met
fixed_matched,Training,Carrot,72,1223,60,1200,0,0,True
fixed_matched,Training,Cabbage,48,921,60,1200,12,279,False
fixed_matched,Training,Potato,77,1127,60,1200,0,73,False
fixed_matched,Validation,Carrot,17,316,20,400,3,84,False
fixed_matched,Validation,Cabbage,12,240,20,400,8,160,False
fixed_matched,Validation,Potato,39,552,20,400,0,0,True
fixed_matched,Reserved test,Carrot,18,56,20,400,2,344,False
fixed_matched,Reserved test,Cabbage,11,93,20,400,9,307,False
fixed_matched,Reserved test,Potato,20,85,20,400,0,315,False


Cohort,Partition,Crop,Usable parcel groups,Usable pixels,Target parcels,Target pixels,Parcel shortfall,Pixel shortfall,Budget met
aligned_matched,Training,Carrot,72,1223,60,1200,0,0,True
aligned_matched,Training,Cabbage,48,921,60,1200,12,279,False
aligned_matched,Training,Potato,77,1127,60,1200,0,73,False
aligned_matched,Validation,Carrot,17,316,20,400,3,84,False
aligned_matched,Validation,Cabbage,12,240,20,400,8,160,False
aligned_matched,Validation,Potato,39,552,20,400,0,0,True
aligned_matched,Reserved test,Carrot,18,56,20,400,2,344,False
aligned_matched,Reserved test,Cabbage,11,93,20,400,9,307,False
aligned_matched,Reserved test,Potato,20,85,20,400,0,315,False


## Training and external-test handoffs

Continue with Training 01–02, then Testing 01–02. The notebooks use the current feature export by default. Select models using validation results only.


In [3]:
print("Development:", feature_handoff.parent / "features_development.json")
print("External test:", feature_handoff.parent / "features_test.json")
print("Readable counts report:", feature_handoff.parent / "preprocessing_report.md")
print("Development and JECAM counts:", feature_handoff.parent / "usable_samples.csv")
print("Test population counts:", feature_handoff.parent / "test_population_coverage.csv")
print("Organized outputs:", root / "pipeline_outputs/preprocessing" / feature_handoff.parent.name / "features")


Development: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/features_development.json
External test: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/features_test.json
Readable counts report: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/preprocessing_report.md
Counts CSV: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/usable_samples.csv
Organized outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/preprocessing/external_jqin0imq/features


## Evidence and interpretation

Coverage describes this saved snapshot. Green-up is estimated timing; sample targets do not establish accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.


## Compare usable coverage

Compare parcels and pixels separately. Cohorts share observations; never sum their counts.


In [ ]:
from crop_pipeline.visualization.demo import plot_usable_samples
from IPython.display import display
import matplotlib.pyplot as plt
figure = plot_usable_samples(usable_samples)
display(figure)
plt.close(figure)


## Usable samples against targets

Dashed lines are the per-crop targets: 60 parcels / 1,200 pixels for training and 20 parcels / 400 pixels each for validation and testing. Bars below a line are shortfalls. Reaching a target does not establish accuracy.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

figure_folder = feature_handoff.parent / "figures"
for cohort in ("fixed_full", "aligned_full", "fixed_matched", "aligned_matched"):
    show_and_save(figures.draw_usable_vs_targets(usable_samples, cohort), figure_folder, f"usable_vs_targets_{cohort}")